# Fault classification with gearbox vibration

Fault classification predicts a discrete health state. GFD provides healthy and
broken-tooth recordings at ten loads, allowing supervised evaluation.

## Learning goals

- convert raw vibration into window-level features;
- split by operating load to test generalization;
- evaluate balanced accuracy and a confusion matrix.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "datasets").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import datasets
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score

In [ ]:
datasets.download("gfd")

def window_features(condition, load, window_size=256):
    frame = datasets.load("gfd", condition=condition, load=load)
    sensors = [column for column in frame.columns if column.startswith("sensor_")]
    return (
        frame.with_row_index("sample")
        .with_columns((pl.col("sample") // window_size).alias("window"))
        .group_by("window")
        .agg(
            *[pl.col(column).mean().alias(f"{column}_mean") for column in sensors],
            *[pl.col(column).std().alias(f"{column}_std") for column in sensors],
            *[
                (pl.col(column) ** 2).mean().sqrt().alias(f"{column}_rms")
                for column in sensors
            ],
        )
        .with_columns(
            pl.lit(condition).alias("condition"),
            pl.lit(load).alias("load"),
        )
    )

features = pl.concat([
    window_features(condition, load)
    for condition in ("healthy", "broken")
    for load in range(0, 100, 10)
])
features.group_by("condition", "load").len().sort("condition", "load")

In [ ]:
predictors = [
    column for column in features.columns
    if column not in {"window", "condition", "load"}
]
train = features.filter(pl.col("load") < 70)
test = features.filter(pl.col("load") >= 70)

classifier = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=0,
)
classifier.fit(train.select(predictors).to_numpy(), train["condition"].to_numpy())
prediction = classifier.predict(test.select(predictors).to_numpy())
print(
    "Held-out-load balanced accuracy:",
    f"{balanced_accuracy_score(test['condition'].to_numpy(), prediction):.3f}",
)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    test["condition"].to_numpy(),
    prediction,
    labels=["healthy", "broken"],
)
plt.title("GFD held-out-load classification")
plt.show()

## Interpretation and limitations

Random row splitting would leak near-identical adjacent vibration samples.
Windowing and holding out complete loads is stricter, but results from one
gearbox should not be interpreted as cross-machine generalization.